# 2-E（課後選做）：上雲端前的準備

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AluminumShark/langgraph-skills-workshop/blob/main/notebooks/2-E_deploy.ipynb)

> 📍 **對應簡報**：第 2 堂「補充：上雲端」，投影片 **45–52**（課堂上不講，課後自學）

## 這本 notebook 在做什麼

前兩堂的作業小助教都住在 Colab 裡：模型是 Colab 內建的、檔案在 `/content`、斷線就沒了。
真的要讓全社團用，就得把它放到「雲端」：一台 24 小時開著、別人用網址就能連到的電腦。

**這本不會真的部署**（那需要雲端帳號和信用卡），而是在免費 Colab 裡，把上雲端前一定要會的四件事各練一次：

| 練習 | 你會做什麼 | 對應投影片 |
| --- | --- | --- |
| 練習 1 | 把雙核小助教包成 **API 服務**，用網址送作業、拿回饋；thread_id 跨請求保留 | 46–48 |
| 練習 2 | 用 **Send** 一次開好幾份「批一份」平行批改，數一數多花了多少 token | 50 |
| 練習 3 | 寄信前用 **interrupt()** 停下來，人按確認才寄 | 51 |
| 練習 4 | **上線前檢查**：金鑰、第三方 Skill、圍欄、部署檔（`Dockerfile`、`langgraph.json`） | 46–47、52 |

## 怎麼用

1. 先執行 **「0. 環境設定」**（會多裝 FastAPI 和 uvicorn，並準備好 2-D 的雙核小助教）。
2. 四個練習彼此獨立，想做哪個就從那個練習的第一格開始。
3. 會呼叫模型好幾次；免費額度不夠時，可以改用講師的 Gemini key（「0. 環境設定」選 `gemini`）。

In [ ]:
#@title 0. 環境設定（第一次打開、或斷線後，先執行這一格）
#@markdown **模型來源**：`colab` ＝ Colab 內建模型（免費、不用 API key）；`gemini` ＝ 講師提供的 Gemini API key（備案）
模型來源 = "colab"  #@param ["colab", "gemini"]
#@markdown 選 `gemini` 時：建議在左邊 🔑「密鑰」新增 `GOOGLE_API_KEY`；也可以把 key 貼在下面（⚠️ 貼了就不要把 notebook 分享出去）
講師_API_KEY = ""  #@param {type:"string"}
#@markdown 進階（通常不用改）：指定模型名稱，留空用預設
colab_模型名稱 = ""  #@param {type:"string"}
gemini_模型名稱 = "gemini-flash-latest"  #@param {type:"string"}

import os, shutil, subprocess, sys
from pathlib import Path

# 1. 用 uv 安裝套件（版本釘死，大約 10–30 秒）
if not os.environ.get("WORKSHOP_SKIP_INSTALL"):
    print("⏳ 用 uv 安裝 LangGraph / LangChain ...")
    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(f'uv pip install --python "{sys.executable}" -q "langgraph>=1.1,<2" "langchain>=1.2,<2" "langchain-google-genai>=4,<5" "langgraph-checkpoint-sqlite>=3,<4" "fastapi>=0.115" "uvicorn>=0.30"', shell=True, check=True)

# 2. 寫出這本 notebook 用到的共用檔案
RB = Path("/content/review-bot")
for d in ['tools', 'deploy', 'skills/friendly-feedback', 'skills/strict-review', 'skills/naming-check/scripts', 'skills/detailed-feedback/scripts']:
    (RB / d).mkdir(parents=True, exist_ok=True)
_檔案 = {
    'llm.py': r'''"""模型轉接頭：把「Colab 內建模型」或「講師的 Gemini API key」包成同一個 model 物件。

學生不用看這個檔案。後面所有 cell 都只寫 model.invoke(...)、model.bind_tools(...)，
不管背後是哪一家模型，用法都一樣。

- colab ：google.colab.ai.generate_text（免費帳號可用，不需要 API key）
- gemini：langchain_google_genai.ChatGoogleGenerativeAI（需要 GOOGLE_API_KEY）

generate_text 只吃一段文字、只回一段文字，不支援原生的 tool calling。
所以 ColabAIChat 做了一個小轉接：
1. 把多則訊息接成一段文字（系統指示、使用者、AI、工具結果各標一個標題）
2. 有綁工具時，在文字最後附上工具清單，請模型「需要工具就只回一行 JSON」
3. 把模型回的 JSON 解析成 tool_calls，格式和原生 tool calling 一模一樣
這剛好讓你看到：模型從頭到尾只是在「說」它想用哪個工具。
"""

from __future__ import annotations

import json
import os
import re
import time
import uuid
from pathlib import Path
from typing import Any

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.utils.function_calling import convert_to_openai_tool

設定檔 = Path(__file__).with_name(".llm.json")
預設_gemini_模型 = "gemini-flash-latest"


def 估算_token(text: str) -> int:
    """粗估 token 數：中日韓文字約 1 字 1 token，其他字元約 4 個 1 token。"""
    cjk = len(re.findall(r"[⺀-鿿豈-﫿＀-￯]", text))
    return max(1, round(cjk + (len(text) - cjk) / 4))


# ---------- 把訊息接成一段文字 ----------

def _工具說明(tools: list[dict]) -> str:
    lines = []
    for t in tools:
        fn = t.get("function", t)
        props = fn.get("parameters", {}).get("properties", {})
        params = ", ".join(f"{k}: {v.get('type', 'string')}" for k, v in props.items())
        desc = " ".join((fn.get("description") or "").split())
        lines.append(f"- {fn['name']}({params})：{desc}")
    return "\n".join(lines)


def _訊息轉文字(messages: list[BaseMessage], tools: list[dict] | None, tool_choice: Any) -> str:
    parts = []
    for m in messages:
        if m.type == "system":
            parts.append(f"【系統指示】\n{m.text}")
        elif m.type == "human":
            parts.append(f"【使用者】\n{m.text}")
        elif m.type == "ai":
            if m.tool_calls:
                calls = [{"name": c["name"], "args": c["args"]} for c in m.tool_calls]
                parts.append("【AI】\n" + json.dumps({"tool_calls": calls}, ensure_ascii=False))
            else:
                parts.append(f"【AI】\n{m.text}")
        elif m.type == "tool":
            name = getattr(m, "name", None) or "工具"
            parts.append(f"【工具 {name} 的結果】\n{m.text}")
    if tools:
        must = ""
        if tool_choice in ("any", "required"):
            must = "這一次你一定要呼叫至少一個工具。\n"
        elif isinstance(tool_choice, str) and tool_choice not in ("auto", "none"):
            must = f"這一次你一定要呼叫 {tool_choice}。\n"
        parts.append(
            "【你可以用的工具】\n"
            f"{_工具說明(tools)}\n\n"
            "【回覆規則】\n"
            f"{must}"
            "如果你需要使用工具，只回一行 JSON，不要加任何其他文字、也不要加 ``` 符號：\n"
            '{"tool_calls": [{"name": "工具名稱", "args": {"參數名稱": "參數值"}}]}\n'
            "如果不需要工具（例如已經有足夠的資訊），就直接用一般文字回答，不要輸出 JSON。"
        )
    parts.append("【AI】")
    return "\n\n".join(parts)


def _解析回覆(text: str, tools: list[dict] | None) -> AIMessage:
    """模型回的是 JSON 紙條就轉成 tool_calls，否則當成一般文字。"""
    meta = {"raw_text": text, "model_provider": "colab"}
    if tools:
        s = text.strip()
        s = re.sub(r"^```(?:json)?\s*|\s*```$", "", s)
        start, end = s.find("{"), s.rfind("}")
        if start != -1 and end > start and ("tool_calls" in s or s.startswith("{")):
            try:
                data = json.loads(s[start : end + 1])
            except json.JSONDecodeError:
                data = None
            calls = None
            if isinstance(data, dict):
                if isinstance(data.get("tool_calls"), list):
                    calls = data["tool_calls"]
                elif "name" in data or "tool" in data:
                    calls = [data]
            if calls:
                tool_calls = []
                for c in calls:
                    if not isinstance(c, dict):
                        continue
                    name = c.get("name") or c.get("tool")
                    args = c.get("args") or c.get("arguments") or {}
                    if isinstance(args, str):
                        try:
                            args = json.loads(args)
                        except json.JSONDecodeError:
                            args = {}
                    if name:
                        tool_calls.append(
                            {"name": name, "args": args, "id": f"call_{uuid.uuid4().hex[:8]}", "type": "tool_call"}
                        )
                if tool_calls:
                    return AIMessage(content="", tool_calls=tool_calls, response_metadata=meta)
    return AIMessage(content=text, response_metadata=meta)


class ColabAIChat(BaseChatModel):
    """把 google.colab.ai.generate_text 包成 LangChain 的聊天模型。"""

    model_name: str | None = None
    max_retries: int = 3

    @property
    def _llm_type(self) -> str:
        return "colab-ai"

    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        formatted = [convert_to_openai_tool(t) for t in tools]
        return self.bind(tools=formatted, tool_choice=tool_choice)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        from google.colab import ai

        tools = kwargs.get("tools")
        prompt = _訊息轉文字(messages, tools, kwargs.get("tool_choice"))
        extra = {"model_name": self.model_name} if self.model_name else {}
        for attempt in range(self.max_retries):
            try:
                text = ai.generate_text(prompt, **extra) or ""
                break
            except Exception:
                if attempt == self.max_retries - 1:
                    raise
                time.sleep(2 * (attempt + 1))
        message = _解析回覆(text, tools)
        message.usage_metadata = {
            "input_tokens": 估算_token(prompt),
            "output_tokens": 估算_token(text),
            "total_tokens": 估算_token(prompt) + 估算_token(text),
        }
        message.response_metadata["token_count"] = "估算值（Colab 內建模型不回報用量）"
        return ChatResult(generations=[ChatGeneration(message=message)])


# ---------- 對外：設定與取得 model ----------

def 讀設定() -> dict:
    try:
        return json.loads(設定檔.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def 存設定(provider: str, colab_model: str = "", gemini_model: str = "") -> None:
    """記住選了哪個模型來源（不存 API key）。"""
    設定檔.write_text(
        json.dumps(
            {"provider": provider, "colab_model": colab_model, "gemini_model": gemini_model or 預設_gemini_模型},
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


def _找_api_key() -> str | None:
    if os.environ.get("GOOGLE_API_KEY"):
        return os.environ["GOOGLE_API_KEY"]
    try:
        from google.colab import userdata

        key = userdata.get("GOOGLE_API_KEY")
        if key:
            os.environ["GOOGLE_API_KEY"] = key
            return key
    except Exception:
        pass
    return None


def get_model() -> BaseChatModel:
    cfg = 讀設定()
    # 上雲端時沒有設定檔：用環境變數 WORKSHOP_PROVIDER=gemini（金鑰從 GOOGLE_API_KEY 讀）
    provider = os.environ.get("WORKSHOP_PROVIDER") or cfg.get("provider", "colab")
    if provider == "gemini":
        key = _找_api_key()
        if not key:
            raise RuntimeError(
                "選了 gemini 但找不到 API key：請在左邊 🔑「密鑰」新增 GOOGLE_API_KEY，"
                "或在「0. 環境設定」的表單貼上講師提供的 key，再重跑那一格。"
            )
        from langchain_google_genai import ChatGoogleGenerativeAI

        return ChatGoogleGenerativeAI(
            model=cfg.get("gemini_model") or 預設_gemini_模型, google_api_key=key, max_retries=3
        )
    try:
        from google.colab import ai  # noqa: F401
    except ImportError as exc:
        raise RuntimeError("google.colab.ai 只能在 Google Colab 上使用；不在 Colab 的話請改選 gemini。") from exc
    return ColabAIChat(model_name=cfg.get("colab_model") or None)
''',
    'workshop.py': r'''"""工作坊 notebook 共用的小工具（學生不用看）。

- 準備()          ：每一段的第一格都會呼叫，重新連上模型、重新載入改過的檔案
- 範例作業         ：表單下拉選單裡的幾段作業
- 記下() / 印出歷程()：把每次執行的 State 存起來，最後一格一次印出來給你截圖
- 顯示流程圖()      ：把 LangGraph 畫成圖
- 存到雲端硬碟() / 下載壓縮檔()
"""

from __future__ import annotations

import datetime as _dt
import importlib
import json
import shutil
import sys
from pathlib import Path

from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import BaseMessage

from llm import get_model, 估算_token  # noqa: F401  （讓 notebook 可以直接 from workshop import 估算_token）

__all__ = [
    "BASE", "準備", "範例作業", "取得作業", "記下", "印出狀態", "印出訂單紙", "印出歷程", "清除歷程",
    "模型呼叫計數器", "顯示流程圖", "存到雲端硬碟", "下載壓縮檔", "估算_token",
]

BASE = Path(__file__).resolve().parent
歷程檔 = BASE / "history.jsonl"

範例作業 = {
    "少了右括號": "print('hi'\n",
    "正確的程式": "def add(a, b):\n    return a + b\n\nprint(add(2, 3))\n",
    "空白": "",
    "變數打錯字": "def add(a, b):\n    total = a + b\n    return summ\n\nprint(add(2, 3))\n",
    "縮排亂掉": "def average(scores):\n  total = 0\n  for s in scores:\n        total += s\n  return total / len(scores)\n\nprint(average([90, 85, 77]))\n",
    "命名不一致": (
        "def CalcAverage(scoreList):\n"
        "    Total = 0\n"
        "    for s in scoreList:\n"
        "        Total += s\n"
        "    avgScore = Total / len(scoreList)\n"
        "    return avgScore\n"
        "\n"
        "myScores = [90, 85, 77]\n"
        "MAX = 100\n"
        "result_2 = CalcAverage(myScores)\n"
        "print(result_2, '/', MAX)\n"
    ),
}


def 取得作業(名稱: str) -> str:
    """表單選「我的作業」時讀 /content/review-bot/我的作業.py，其他從範例裡拿。"""
    if 名稱.startswith("我的作業"):
        path = BASE / "我的作業.py"
        if not path.exists():
            raise FileNotFoundError("還沒有「我的作業」：請先執行上面寫著 %%writefile ... 我的作業.py 的那一格。")
        return path.read_text(encoding="utf-8")
    return 範例作業[名稱]


def 準備():
    """每一段的起點：把 review-bot 放進 import 路徑、重新載入改過的檔案、回傳 model。"""
    if str(BASE) not in sys.path:
        sys.path.insert(0, str(BASE))
    importlib.invalidate_caches()
    for name, mod in list(sys.modules.items()):
        path = getattr(mod, "__file__", None) or ""
        if path and Path(path).resolve().is_relative_to(BASE) and name not in ("workshop", "llm"):
            try:
                importlib.reload(mod)
            except Exception:
                sys.modules.pop(name, None)
    return get_model()


# ---------- State 歷程 ----------

def _可存的(obj):
    if isinstance(obj, BaseMessage):
        d = {"type": obj.type, "content": obj.text}
        if getattr(obj, "tool_calls", None):
            d["tool_calls"] = [{"name": c["name"], "args": c["args"]} for c in obj.tool_calls]
        if obj.type == "tool":
            d["name"] = getattr(obj, "name", None)
        return d
    if isinstance(obj, dict):
        return {str(k): _可存的(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_可存的(v) for v in obj]
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    return str(obj)


def 記下(標題: str, 內容) -> None:
    """把這一次的結果（通常是 State）存進 history.jsonl。"""
    row = {"時間": _dt.datetime.now().strftime("%H:%M:%S"), "標題": 標題, "內容": _可存的(內容)}
    with 歷程檔.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def _短(text, n=300):
    text = str(text)
    return text if len(text) <= n else text[:n] + f"…（還有 {len(text) - n} 字）"


def 印出狀態(state: dict, 縮排: str = "  ") -> None:
    for k, v in state.items():
        if k == "messages":
            print(f"{縮排}{k}: （{len(v)} 則訊息）")
            for m in v:
                m = _可存的(m)
                if isinstance(m, dict) and m.get("tool_calls"):
                    print(f"{縮排}  [{m['type']}] 呼叫 {m['tool_calls']}")
                elif isinstance(m, dict):
                    print(f"{縮排}  [{m.get('type')}] {_短(m.get('content', ''), 200)}")
        else:
            print(f"{縮排}{k}: {_短(repr(v) if not isinstance(v, str) else repr(v), 300)}")


def 印出訂單紙(第幾步: int, state: dict, 上一步: dict | None = None, 不印: tuple = ("code",)) -> None:
    """stream_mode="values" 每一步都會給整張訂單紙；順便標出這一步新寫了哪幾格。"""
    if not 上一步:
        說明 = "起點：空白訂單紙"
    else:
        新寫入 = [k for k in state if state.get(k) != 上一步.get(k)]
        說明 = "新寫入：" + ("、".join(新寫入) if 新寫入 else "（沒有改變）")
    print(f"第 {第幾步} 步（{說明}）")
    print("  ", {k: (_短(v, 120) if isinstance(v, str) else v) for k, v in state.items() if k not in 不印})


def 印出歷程() -> None:
    if not 歷程檔.exists():
        print("還沒有任何紀錄：先跑上面幾段，再回來執行這一格。")
        return
    rows = [json.loads(line) for line in 歷程檔.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"📸 共 {len(rows)} 筆紀錄（存在 {歷程檔}）\n")
    for i, row in enumerate(rows, 1):
        print(f"===== {i}. [{row['時間']}] {row['標題']} =====")
        內容 = row["內容"]
        if isinstance(內容, dict):
            印出狀態(內容)
        elif isinstance(內容, list):
            for j, item in enumerate(內容, 1):
                print(f"  --- 第 {j} 步 ---")
                if isinstance(item, dict):
                    印出狀態(item, "    ")
                else:
                    print(f"    {_短(item)}")
        else:
            print(f"  {_短(內容, 1000)}")
        print()


def 清除歷程() -> None:
    歷程檔.unlink(missing_ok=True)


# ---------- 數模型被呼叫了幾次 ----------

class 模型呼叫計數器(BaseCallbackHandler):
    """放進 config={"callbacks": [計數器]}，每呼叫一次模型就 +1。"""

    def __init__(self):
        self.次數 = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.次數 += 1

    def on_llm_start(self, serialized, prompts, **kwargs):
        self.次數 += 1


# ---------- 畫流程圖 ----------

def 顯示流程圖(graph) -> None:
    g = graph.get_graph()
    try:
        from IPython.display import Image, display

        display(Image(g.draw_mermaid_png()))
    except Exception:
        print("（畫圖服務暫時連不上，改用文字版流程圖；可以貼到 https://mermaid.live 看圖）\n")
        print(g.draw_mermaid())


# ---------- 保存 ----------

def 存到雲端硬碟() -> None:
    from google.colab import drive

    drive.mount("/content/drive")
    目的地 = Path("/content/drive/MyDrive/langgraph-skills-workshop") / _dt.datetime.now().strftime("%Y%m%d-%H%M%S")
    shutil.copytree(BASE, 目的地, ignore=shutil.ignore_patterns("__pycache__", ".llm.json"))
    print(f"✅ 已存到雲端硬碟：{目的地}")
    print("   （notebook 本身請用上方選單「檔案 → 在雲端硬碟中儲存副本」）")


def 下載壓縮檔() -> None:
    from google.colab import files

    名稱 = BASE.parent / ("review-bot-" + _dt.datetime.now().strftime("%Y%m%d-%H%M%S"))
    zip_path = shutil.make_archive(str(名稱), "zip", BASE.parent, BASE.name)
    print(f"✅ 已打包：{zip_path}")
    files.download(zip_path)
''',
    'tools/__init__.py': r'''"""作業小助教的工具箱：run_code、check_indent、read_spec、check_naming。"""
''',
    'tools/run_code.py': r'''"""run_code：在籠子裡執行學生的作業。

學生作業裡什麼都可能有（import os 刪檔、無限迴圈），所以這個工具有三道圍欄：
1. 執行前先檢查：不准 import os / sys / subprocess / shutil 這類危險模組，
   也不准用 eval、exec、open 和底線開頭的屬性（常被拿來繞過限制）
2. 開一個獨立的子程序執行，超過時間就強制停止（無限迴圈也停得下來）
3. 只回一行結果（第幾行、什麼錯），模型才看得懂

⚠️ 這是教學用的圍欄。正式產品請用真正的 sandbox（容器、RestrictedPython、e2b、Modal）。
"""

import ast
import json
import os
import subprocess
import sys
import tempfile

from langchain.tools import tool

禁止的模組 = {"os", "sys", "subprocess", "shutil", "socket", "pathlib", "ctypes", "importlib",
           "multiprocessing", "threading", "signal", "pickle", "builtins", "io", "inspect", "gc"}
允許的模組 = {"math", "random", "string", "re", "datetime", "collections", "itertools", "functools",
           "statistics", "time", "decimal", "fractions", "json", "copy", "heapq", "bisect"}
禁止的函式 = {"eval", "exec", "compile", "open", "input", "__import__", "globals", "locals", "vars",
           "getattr", "setattr", "delattr", "breakpoint", "help", "exit", "quit", "memoryview"}
禁止的屬性 = {"system", "popen", "modules", "builtins", "f_globals", "f_back", "gi_frame", "cr_frame"}
最長時間_秒 = 3
最長輸出_字 = 800

# 子程序裡真正執行作業的那段程式：用「縮小版的 builtins」執行，import 只放行白名單
_RUNNER = r"""
import builtins, contextlib, io, json, sys, traceback
code = sys.stdin.buffer.read().decode("utf-8")
allowed = set(json.loads(sys.argv[1]))
blocked = set(json.loads(sys.argv[2]))
real_import = builtins.__import__
def guarded_import(name, globals=None, locals=None, fromlist=(), level=0):
    if name.split(".")[0] not in allowed:
        raise ImportError("這個工具不允許 import " + name)
    return real_import(name, globals, locals, fromlist, level)
safe = {k: v for k, v in vars(builtins).items() if k not in blocked}
safe["__import__"] = guarded_import
try:
    import resource
    resource.setrlimit(resource.RLIMIT_AS, (512 * 1024 * 1024, 512 * 1024 * 1024))
except Exception:
    pass
out = io.StringIO()
result = {"ok": True}
try:
    compiled = compile(code, "作業.py", "exec")
    with contextlib.redirect_stdout(out):
        exec(compiled, {"__builtins__": safe, "__name__": "__main__"})
except SyntaxError as e:
    result = {"ok": False, "line": e.lineno, "error": "SyntaxError: " + str(e.msg)}
except BaseException as e:
    line = None
    for frame in traceback.extract_tb(e.__traceback__):
        if frame.filename == "作業.py":
            line = frame.lineno
    result = {"ok": False, "line": line, "error": type(e).__name__ + ": " + str(e)}
result["stdout"] = out.getvalue()
sys.__stdout__.write(json.dumps(result))
"""


def _先檢查(code: str) -> str | None:
    """執行前的靜態檢查；有問題就回一行說明，沒問題回 None。"""
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f"❌ 第 {e.lineno} 行 SyntaxError: {e.msg}"
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            names = [a.name for a in node.names] if isinstance(node, ast.Import) else [node.module or ""]
            for name in names:
                root = name.split(".")[0]
                if root in 禁止的模組 or root not in 允許的模組:
                    return f"⛔ 已擋下：第 {node.lineno} 行 import {name}（這個工具不允許使用 {root}）"
        elif isinstance(node, ast.Name) and node.id in 禁止的函式:
            return f"⛔ 已擋下：第 {node.lineno} 行用了 {node.id}（這個工具不允許）"
        elif isinstance(node, ast.Attribute) and (
            node.attr.startswith("_") or node.attr in 禁止的屬性 or node.attr in 禁止的模組
        ):
            return f"⛔ 已擋下：第 {node.lineno} 行存取了 .{node.attr}（這個工具不允許）"
    return None


def 執行作業(code: str, 最長時間: float = 最長時間_秒) -> str:
    """真正做事的函式：檢查 → 子程序執行 → 回傳一行結果。"""
    if not code.strip():
        return "❌ 作業是空白的，沒有東西可以執行"
    擋下 = _先檢查(code)
    if 擋下:
        return 擋下
    env = {"PYTHONIOENCODING": "utf-8", "PATH": os.environ.get("PATH", "")}
    if os.name == "nt":
        env["SYSTEMROOT"] = os.environ.get("SYSTEMROOT", "")
    try:
        proc = subprocess.run(
            [sys.executable, "-I", "-c", _RUNNER, json.dumps(sorted(允許的模組)), json.dumps(sorted(禁止的函式))],
            input=code, capture_output=True, text=True, encoding="utf-8",
            timeout=最長時間, cwd=tempfile.gettempdir(), env=env,
        )
    except subprocess.TimeoutExpired:
        return f"⏱️ 執行超過 {最長時間:g} 秒被強制停止（可能有無限迴圈）"
    try:
        result = json.loads(proc.stdout.strip().splitlines()[-1])
    except (IndexError, json.JSONDecodeError):
        return "❌ 執行失敗（程式沒有正常結束，可能用了太多記憶體）"
    stdout = result.get("stdout", "")
    if len(stdout) > 最長輸出_字:
        stdout = stdout[:最長輸出_字] + "\n…（輸出太長，後面省略）"
    if result["ok"]:
        return f"✅ 執行成功，輸出：\n{stdout}" if stdout else "✅ 執行成功（沒有印出任何東西）"
    where = f"第 {result['line']} 行 " if result.get("line") else ""
    印出 = f"\n（出錯前印出：{stdout.strip()}）" if stdout.strip() else ""
    return f"❌ {where}{result['error']}{印出}"


@tool
def run_code(code: str) -> str:
    """安全地執行一段 Python 作業，回傳輸出或錯誤訊息。

    什麼時候用：想知道作業「實際跑起來」會印出什麼、會不會出錯時。
    參數 code：完整的 Python 程式碼（字串）。
    回傳：一行結果，例如「✅ 執行成功，輸出：5」或「❌ 第 3 行 NameError: ...」。
    限制：不能 import os、sys、subprocess 等模組；超過 3 秒會被停止。
    """
    return 執行作業(code)
''',
    'tools/read_spec.py': r'''"""read_spec：讀這一題的作業要求。

題目要求存在 review-bot/spec.md，notebook 的表單改了題目就會重寫這個檔案。
"""

from pathlib import Path

from langchain.tools import tool

題目檔 = Path(__file__).resolve().parent.parent / "spec.md"


def 讀題目() -> str:
    if not 題目檔.exists():
        return "（這一題沒有提供作業要求）"
    return 題目檔.read_text(encoding="utf-8").strip() or "（這一題沒有提供作業要求）"


@tool
def read_spec() -> str:
    """讀取這一題的作業要求（題目寫了什麼、要交什麼）。

    什麼時候用：要判斷作業有沒有做到題目要求之前，例如函式名稱、輸出格式對不對。
    不需要參數。
    回傳：題目要求的全文。
    """
    return 讀題目()
''',
    'tools/check_naming.py': r'''"""check_naming：把 Skill 的腳本 scripts/check_naming.py 包成 agent 的工具。

腳本外面包一層 @tool，就進了 agent 的工具箱：
AI 只負責「決定要不要跑」和「把結果抄進回饋」，檢查本身是 Python，每次都一樣。
"""

import subprocess
import sys
from pathlib import Path

from langchain.tools import tool

腳本 = Path(__file__).resolve().parent.parent / "skills" / "naming-check" / "scripts" / "check_naming.py"


def 執行命名腳本(code: str) -> str:
    proc = subprocess.run(
        [sys.executable, str(腳本)], input=code.encode("utf-8"), capture_output=True, timeout=10
    )
    out = proc.stdout.decode("utf-8").strip()
    return out or "腳本執行失敗：" + proc.stderr.decode("utf-8", "replace").strip()[-200:]


@tool
def check_naming(code: str) -> str:
    """檢查變數與函式名稱是否為小寫加底線（snake_case），回傳不合規則的行。

    什麼時候用：批改作業、需要列出命名問題時（這是 naming-check 手冊附的腳本）。
    參數 code：完整的 Python 程式碼（字串）。
    回傳：每個問題一行，例如「第 3 行：myVar 不符合規則（建議 my_var）」；全部合格時回傳「OK：…」。
    """
    return 執行命名腳本(code)
''',
    'graph.py': r'''"""1-A：作業小助教的 workflow，路線由程式決定。

START → check_syntax ─語法 OK→ give_feedback → check_format ─格式 OK→ END
                     └語法錯→ explain_error → END     └不合格、retries 還沒到上限→ 退回 give_feedback

三個零件：
- State：一張訂單紙（dict），從頭傳到尾
- Node ：一個工作站（函式），收 State、只回傳自己改的那幾格
- Edge ：工作站之間的箭頭；條件邊就是一個回傳「下一站名字」的 if
"""

import ast
import re
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class State(TypedDict, total=False):
    code: str              # 學生交的作業
    syntax_ok: bool | None # 語法對不對（還沒檢查時是 None）
    error: str             # 語法錯在哪裡
    feedback: str          # 給學生的回饋
    format_ok: bool | None # 回饋格式合不合格
    format_problem: str    # 不合格的原因（退回重寫時會告訴 AI）
    retries: int           # 回饋寫了幾次


# ---------- Node 1：check_syntax（純 Python） ----------

_錯誤翻譯 = [
    (r"'(.)' was never closed", lambda m: f"少了右括號 {dict(zip('([{', ')]}')).get(m.group(1), '')}"),
    (r"unterminated string literal", lambda m: "字串少了結尾的引號"),
    (r"expected ':'", lambda m: "少了冒號 :"),
    (r"unexpected indent", lambda m: "縮排多了"),
    (r"expected an indented block", lambda m: "這裡需要縮排"),
    (r"unindent does not match", lambda m: "縮排對不齊"),
    (r"unmatched '(.)'", lambda m: f"多了一個 {m.group(1)}"),
    (r"invalid character", lambda m: "有全形或看不懂的字元"),
    (r"Perhaps you forgot a comma", lambda m: "可能少了逗號"),
    (r"invalid syntax", lambda m: "語法不正確"),
]


def 翻譯語法錯誤(msg: str) -> str:
    for pattern, 說法 in _錯誤翻譯:
        m = re.search(pattern, msg)
        if m:
            return 說法(m)
    return msg


def check_syntax(state: State) -> dict:
    code = state.get("code", "")
    if not code.strip():
        return {"syntax_ok": False, "error": "作業是空白的，沒有任何程式"}
    try:
        ast.parse(code)
    except SyntaxError as e:
        return {"syntax_ok": False, "error": f"第 {e.lineno} 行{翻譯語法錯誤(e.msg)}"}
    return {"syntax_ok": True, "error": ""}


# ---------- 條件邊 1：語法對 → 給回饋；語法錯 → 解釋錯誤 ----------

def route_after_syntax(state: State) -> str:
    if state["syntax_ok"]:
        return "give_feedback"
    return "explain_error"


# ---------- Node 2：explain_error（純 Python） ----------

def explain_error(state: State) -> dict:
    return {"feedback": f"語法：{state['error']}\n改進：先把這一行修好，再執行一次看看。"}


# ---------- Node 3：give_feedback（兩個版本） ----------

def give_feedback_固定文字(state: State) -> dict:
    """先不接 AI 的版本：回固定的一段話。"""
    return {
        "feedback": "優點：語法正確，程式可以執行。\n改進：（這一版還沒接 AI，先用固定文字）",
        "retries": state.get("retries", 0) + 1,
    }


def 建立_give_feedback(model, 回饋字數上限: int):
    """接上 AI 的版本：AI 只是住在這個節點裡的一行程式。"""

    def give_feedback(state: State) -> dict:
        prompt = (
            "你是 Python 作業小助教，請用繁體中文給下面這段作業回饋。\n"
            "格式規定（一定要遵守）：\n"
            "- 只寫兩段：第一段以「優點：」開頭，第二段以「改進：」開頭\n"
            f"- 兩段加起來不超過 {回饋字數上限} 個字\n\n"
            f"作業：\n```python\n{state['code']}\n```"
        )
        if state.get("format_problem"):
            prompt += f"\n\n你上一版的回饋被退回，原因：{state['format_problem']}。請修正後重寫。"
        text = model.invoke(prompt).text
        return {"feedback": text.strip(), "retries": state.get("retries", 0) + 1}

    return give_feedback


# ---------- Node 4：check_format（純 Python 的檢查站） ----------

def 建立_check_format(回饋字數上限: int):
    def check_format(state: State) -> dict:
        feedback = state.get("feedback", "")
        問題 = []
        if "優點" not in feedback:
            問題.append("少了「優點」段落")
        if "改進" not in feedback:
            問題.append("少了「改進」段落")
        if len(feedback) > 回饋字數上限:
            問題.append(f"太長了：{len(feedback)} 字，上限 {回饋字數上限} 字")
        return {"format_ok": not 問題, "format_problem": "；".join(問題)}

    return check_format


# ---------- 條件邊 2：格式 OK 或重寫次數到了 → 結束；否則退回重寫 ----------

def 建立_route_after_format(最多重試次數: int):
    def route_after_format(state: State) -> str:
        if state["format_ok"]:
            return END
        if state.get("retries", 0) >= 最多重試次數:
            return END  # 重寫次數到了就停，不然會無限轉圈
        return "give_feedback"

    return route_after_format


# ---------- 把零件接起來 ----------

def 建立小助教(model=None, 最多重試次數: int = 3, 回饋字數上限: int = 150):
    """model=None 時 give_feedback 用固定文字（先不接 AI）。"""
    give_feedback = 建立_give_feedback(model, 回饋字數上限) if model else give_feedback_固定文字

    builder = StateGraph(State)
    builder.add_node("check_syntax", check_syntax)
    builder.add_node("explain_error", explain_error)
    builder.add_node("give_feedback", give_feedback)
    builder.add_node("check_format", 建立_check_format(回饋字數上限))

    builder.add_edge(START, "check_syntax")
    builder.add_conditional_edges("check_syntax", route_after_syntax, ["give_feedback", "explain_error"])
    builder.add_edge("explain_error", END)
    builder.add_edge("give_feedback", "check_format")
    builder.add_conditional_edges("check_format", 建立_route_after_format(最多重試次數), ["give_feedback", END])
    return builder.compile()


def 空白訂單紙(code: str) -> State:
    return {"code": code, "syntax_ok": None, "error": "", "feedback": "", "format_ok": None,
            "format_problem": "", "retries": 0}
''',
    'skill_loader.py': r'''"""skill_loader：AI 選手冊只看「名片」，選中了才翻開內文。

漸進式揭露三層：
1. 書背（名片）：SKILL.md 開頭兩條 --- 之間的 name 和 description，約 100 token，一直在桌上
2. 內文：整份 SKILL.md，被選中才讀
3. 附錄：scripts/、references/ 裡的檔案，用到才打開
"""

from pathlib import Path

SKILLS = Path(__file__).resolve().parent / "skills"


def 讀名片(path: Path) -> dict:
    """只讀兩條 --- 之間的 name 和 description。"""
    text = Path(path).read_text(encoding="utf-8")
    card = {"name": Path(path).parent.name, "description": "", "path": str(path)}
    if text.startswith("---"):
        head = text.split("---", 2)[1]
        for line in head.strip().splitlines():
            key, _, value = line.partition(":")
            if key.strip() in ("name", "description"):
                card[key.strip()] = value.strip()
    return card


def 讀內文(card: dict) -> str:
    """選中之後才翻開：名片以下的手冊內容。"""
    text = Path(card["path"]).read_text(encoding="utf-8")
    return text.split("---", 2)[2].strip() if text.startswith("---") else text


def 列出名片(只要: list[str] | None = None) -> list[dict]:
    cards = [讀名片(p) for p in sorted(SKILLS.glob("*/SKILL.md"))]
    if 只要:
        cards = [c for c in cards if Path(c["path"]).parent.name in 只要]
    return cards


def 挑手冊(model, 需求: str, cards: list[dict]) -> dict:
    """把所有名片給模型看，讓它挑一本；回傳被挑中的那張名片。"""
    清單 = "\n".join(f"- {c['name']}：{c['description']}" for c in cards)
    回答 = model.invoke(
        "你要替下面的批改需求挑一本最適合的手冊。\n"
        f"批改需求：{需求}\n\n"
        f"手冊名片：\n{清單}\n\n"
        "只回答手冊的 name，不要其他字。"
    ).text.strip()
    提到的 = [(回答.find(c["name"]), c) for c in cards if c["name"] in 回答]
    return min(提到的, key=lambda x: x[0])[1] if 提到的 else cards[0]
''',
    'dual_core.py': r'''"""2-D：雙核作業小助教。外面是 1-A 的 workflow（順序鎖死），裡面是 1-B 的 agent（放手判斷）。

START → check_syntax ─語法 OK→ skill_loader → give_feedback → check_format ─OK→ END
                     └語法錯→ explain_error → END            └不合格→ 退回 give_feedback

- skill_loader ：看所有手冊的名片，挑一本，把內文放進 State（2-A 的手冊）
- give_feedback：裡面是一個小 agent，用挑中的手冊當 system prompt，
                 工具箱有 run_code、read_spec、check_naming（2-B 的腳本包成工具）
- checkpointer ：每一步的 State 都存進 SQLite 檔，同一個 thread_id 可以接著跑
"""

import operator
import sqlite3
from pathlib import Path
from typing import Annotated, TypedDict

from langchain.agents import create_agent
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph

from graph import check_syntax, explain_error, route_after_syntax, 建立_check_format, 建立_route_after_format
from skill_loader import 列出名片, 挑手冊, 讀內文
from tools.check_naming import check_naming
from tools.read_spec import read_spec
from tools.run_code import run_code

存檔位置 = Path(__file__).resolve().parent / "checkpoints.sqlite"
可選的手冊 = ["friendly-feedback", "strict-review"]


class 雙核State(TypedDict, total=False):
    code: str                 # 學生交的作業
    request: str              # 批改需求（skill_loader 用它挑手冊）
    syntax_ok: bool | None
    error: str
    skill_name: str           # 挑中的手冊
    skill: str                # 手冊內文
    feedback: str
    format_ok: bool | None
    format_problem: str
    retries: int
    tools_used: Annotated[list[str], operator.add]  # 裡面的 agent 用過哪些工具（往下接著寫）


def 建立_skill_loader(model):
    def skill_loader(state: 雙核State) -> dict:
        card = 挑手冊(model, state.get("request", ""), 列出名片(可選的手冊))
        return {"skill_name": card["name"], "skill": 讀內文(card)}

    return skill_loader


def 建立_give_feedback(model, 回饋字數上限: int):
    def give_feedback(state: 雙核State) -> dict:
        system_prompt = (
            f"{state['skill']}\n\n"
            "# 你的工具\n"
            "- check_naming：檢查變數命名（手冊附的腳本），要談命名時用它，照抄它的結果\n"
            "- run_code：執行作業看輸出\n"
            "- read_spec：讀題目要求\n"
            "要不要用、用哪一個，你自己決定。\n\n"
            "# 格式（外層流程會用程式檢查，不合格會退回）\n"
            "- 只寫兩段：第一段以「優點：」開頭，第二段以「改進：」開頭\n"
            f"- 兩段加起來不超過 {回饋字數上限} 個字"
        )
        if state.get("format_problem"):
            system_prompt += f"\n\n上一版被退回，原因：{state['format_problem']}。請修正。"
        小agent = create_agent(model, tools=[check_naming, run_code, read_spec], system_prompt=system_prompt)
        result = 小agent.invoke(
            {"messages": [{"role": "user", "content": f"請批改這份作業：\n```python\n{state['code']}\n```"}]},
            config={"recursion_limit": 25},
        )
        messages = result["messages"]
        used = [c["name"] for m in messages for c in (getattr(m, "tool_calls", None) or [])]
        return {"feedback": messages[-1].text.strip(), "retries": state.get("retries", 0) + 1, "tools_used": used}

    return give_feedback


def 逐站執行(小助教, 輸入, config: dict) -> dict:
    """一站一站跑，印出每一站寫了哪幾格；回傳跑完（或中斷）時存檔裡的 State。"""
    for event in 小助教.stream(輸入, config=config, stream_mode="updates"):
        for 站, 寫入 in event.items():
            if 站 == "__interrupt__":
                print("⏸️  中斷：流程停在這裡，State 已經存進存檔")
                continue
            print(f"▶ {站}")
            for key, value in (寫入 or {}).items():
                text = value if isinstance(value, str) else repr(value)
                if key == "skill":
                    text = text.splitlines()[0] + " …（手冊內文）"
                print(f"    {key}: {text[:200]}")
    return 小助教.get_state(config).values


def 開存檔(path: Path = 存檔位置) -> SqliteSaver:
    """把 State 存進 SQLite 檔：重跑 cell、甚至重新啟動執行階段，存檔都還在。"""
    return SqliteSaver(sqlite3.connect(str(path), check_same_thread=False))


def 建立雙核小助教(model, checkpointer=None, 最多重試次數: int = 3, 回饋字數上限: int = 250,
                interrupt_before: list[str] | None = None):
    builder = StateGraph(雙核State)
    builder.add_node("check_syntax", check_syntax)
    builder.add_node("explain_error", explain_error)
    builder.add_node("skill_loader", 建立_skill_loader(model))
    builder.add_node("give_feedback", 建立_give_feedback(model, 回饋字數上限))
    builder.add_node("check_format", 建立_check_format(回饋字數上限))

    builder.add_edge(START, "check_syntax")
    # 語法 OK 先去挑手冊，再給回饋
    builder.add_conditional_edges(
        "check_syntax", route_after_syntax, {"give_feedback": "skill_loader", "explain_error": "explain_error"}
    )
    builder.add_edge("explain_error", END)
    builder.add_edge("skill_loader", "give_feedback")
    builder.add_edge("give_feedback", "check_format")
    builder.add_conditional_edges("check_format", 建立_route_after_format(最多重試次數), ["give_feedback", END])
    return builder.compile(checkpointer=checkpointer, interrupt_before=interrupt_before)
''',
    'spec.md': r'''請寫一個函式 add(a, b)，回傳兩個數字相加的結果，並用 print(add(2, 3)) 印出 5。
''',
    'deploy/__init__.py': r'''"""2-E：上雲端前的準備（API 服務、平行批改、寄信前人工確認、上線前檢查）。"""
''',
    'skills/friendly-feedback/SKILL.md': r'''---
name: friendly-feedback
description: 給 Python 初學者的作業回饋。語氣溫和，先講優點再講改進。適合第一次交作業、剛學會 Python 的同學。
---

# 溫和回饋規則

1. 先講一個做得好的地方
2. 最多指出兩個問題，每個都附上改法
3. 不使用「錯誤」「很差」這類字眼
4. 回饋分兩段：第一段以「優點：」開頭，第二段以「改進：」開頭
''',
    'skills/strict-review/SKILL.md': r'''---
name: strict-review
description: 期中考、正式評分用的嚴格審查。列出所有問題、不留情面、附上扣分。
---

# 嚴格審查規則

1. 列出所有問題，一個都不放過，每個問題都標出第幾行
2. 不說客套話，不用「很棒」「不錯」這類字眼
3. 每個問題附上扣幾分（滿分 100 分）
4. 回饋分兩段：第一段以「優點：」開頭（真的沒有就寫「優點：無」），第二段以「改進：」開頭
''',
    'skills/naming-check/SKILL.md': r'''---
name: naming-check
description: 檢查 Python 作業的變數與函式名稱是不是小寫英文加底線（snake_case）。批改作業、需要列出命名問題時使用。
---

# 命名檢查

執行 scripts/check_naming.py（工具名稱 check_naming），把它的輸出寫進回饋。
不要自己判斷、不要增加或刪掉任何一項，照抄它的每一行。

輸出格式：每個問題一行，例如「第 3 行：myVar」
''',
    'skills/naming-check/scripts/check_naming.py': r'''"""檢查變數與函式名稱是不是小寫英文加底線（snake_case）。

用法：
    python check_naming.py 作業.py        # 檢查一個檔案
    python check_naming.py < 作業.py      # 或從標準輸入讀
輸出：
    每個問題一行，例如「第 3 行：myVar 不符合規則（建議 my_var）」
    全部合格時印「OK：所有變數與函式名稱都是小寫加底線」

一個腳本只做一件事、輸出短短幾行，AI 才看得懂；每次執行結果都一模一樣。
"""

import ast
import re
import sys

規則 = re.compile(r"^_?[a-z][a-z0-9_]*$")


def 建議名稱(name: str) -> str:
    s = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", name)
    return re.sub(r"_+", "_", s).lower()


def 找出所有名稱(tree: ast.AST) -> dict[str, int]:
    """名稱 → 第一次出現的行號（變數、函式、參數，不含 class 名稱）。"""
    名稱 = {}

    def 記(name, line):
        if name not in 名稱 or line < 名稱[name]:
            名稱[name] = line

    for node in ast.walk(tree):
        if isinstance(node, ast.Name) and isinstance(node.ctx, ast.Store):
            記(node.id, node.lineno)
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            記(node.name, node.lineno)
        elif isinstance(node, ast.arg) and node.arg not in ("self", "cls"):
            記(node.arg, node.lineno)
    return 名稱


def 檢查命名(code: str) -> str:
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f"無法檢查：第 {e.lineno} 行有語法錯誤"
    問題 = [
        f"第 {line} 行：{name} 不符合規則（建議 {建議名稱(name)}）"
        for name, line in sorted(找出所有名稱(tree).items(), key=lambda x: (x[1], x[0]))
        if not 規則.match(name)
    ]
    return "\n".join(問題) if 問題 else "OK：所有變數與函式名稱都是小寫加底線"


if __name__ == "__main__":
    if len(sys.argv) > 1:
        with open(sys.argv[1], encoding="utf-8") as f:
            source = f.read()
    else:
        source = sys.stdin.buffer.read().decode("utf-8")
    sys.stdout.reconfigure(encoding="utf-8")
    print(檢查命名(source))
''',
}
for rel, content in _檔案.items():
    (RB / rel).write_text(content, encoding="utf-8")
sys.path.insert(0, str(RB))

# 3. 記住模型來源（不會存 API key），測試連線
import llm
llm.存設定(模型來源, colab_模型名稱.strip(), gemini_模型名稱.strip())
if 講師_API_KEY.strip():
    os.environ["GOOGLE_API_KEY"] = 講師_API_KEY.strip()
from workshop import *
model = 準備()

回覆 = model.invoke("用一句話跟剛學 Python 的同學打招呼").text
print(f"✅ 環境設定完成，模型來源：{模型來源}")
print(f"🤖 模型說：{回覆.strip()}")

---
## 練習 1：把小助教包成一個 API 服務

> 📍 **對應簡報**：投影片 46「離開 Colab：三樣東西要換掉」、47「打包」、48「放到哪一朵雲」

上雲端之後，雲端機器上**真正在跑的**就是一個「服務」：它一直開著，等別人用網址（HTTP）把作業送進來，再把回饋送回去。
送作業的可以是網頁表單、LINE bot、或每天半夜的排程。

下一格寫出 `deploy/server.py`，裡面用 FastAPI 開了三個網址：

| 網址 | 做什麼 |
| --- | --- |
| `GET /health` | 雲端平台定時敲它，確認服務還活著 |
| `POST /review` | 送作業進來（附上 thread_id），拿回饋 |
| `GET /threads/{thread_id}` | 查某個 thread_id 的存檔 |

### 🤔 預測題
1. 用 thread_id「小明-期中」送一次作業後，再去查「小明-期中」和「還沒交作業的同學」的存檔，各會看到什麼？
2. 這個服務關掉再開，「小明-期中」的存檔還在嗎？（提示：存檔放在哪裡？）

In [ ]:
%%writefile /content/review-bot/deploy/server.py
"""練習 1：把雙核小助教包成一個網路服務（API）。

上雲端之後，雲端機器上真正在跑的就是這種程式：
別人（網頁、LINE bot、排程）用 HTTP 把作業送進來，它把回饋送回去。
每個學生、每份作業用一個 thread_id 記住進度，下次用同一個 thread_id 就接得回來。

本機或容器裡啟動：uvicorn deploy.server:app --port 8080
"""

from fastapi import FastAPI
from pydantic import BaseModel

import dual_core
from llm import get_model

app = FastAPI(title="作業小助教 API")
_小助教 = None


def 小助教():
    """第一次有人呼叫時才建立（模型、存檔都只建一次）。"""
    global _小助教
    if _小助教 is None:
        _小助教 = dual_core.建立雙核小助教(get_model(), checkpointer=dual_core.開存檔())
    return _小助教


class 批改請求(BaseModel):
    thread_id: str
    code: str
    request: str = "幫第一次交作業的初學者看看"


@app.get("/health")
def health() -> dict:
    """雲端平台會定時敲這個網址，確認服務還活著。"""
    return {"ok": True}


@app.post("/review")
def review(req: 批改請求) -> dict:
    config = {"configurable": {"thread_id": req.thread_id}}
    state = 小助教().invoke({"code": req.code, "request": req.request, "retries": 0}, config)
    return {
        "thread_id": req.thread_id,
        "skill": state.get("skill_name"),
        "feedback": state.get("feedback"),
        "format_ok": state.get("format_ok"),
        "retries": state.get("retries"),
    }


@app.get("/threads/{thread_id}")
def get_thread(thread_id: str) -> dict:
    """查某個 thread_id 的存檔：State 有沒有保留下來。"""
    snap = 小助教().get_state({"configurable": {"thread_id": thread_id}})
    return {
        "thread_id": thread_id,
        "有存檔": bool(snap.values),
        "下一站": list(snap.next),
        "State": {k: v for k, v in snap.values.items() if k not in ("skill", "code")},
    }


In [ ]:
#@title ▶ 執行：在 Colab 裡把小助教開成 API 服務
import sys; sys.path.insert(0, "/content/review-bot")
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
import socket, threading, time
import requests, uvicorn
from deploy import server

舊的 = getattr(sys, "_小助教伺服器", None)       # 重跑這一格時，先把上一次開的關掉
if 舊的:
    舊的.should_exit = True
    time.sleep(1)

with socket.socket() as s:                       # 找一個沒人用的 port
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]
伺服器 = uvicorn.Server(uvicorn.Config(server.app, host="127.0.0.1", port=port, log_level="warning"))
threading.Thread(target=伺服器.run, daemon=True).start()
sys._小助教伺服器 = 伺服器

for _ in range(50):
    try:
        if requests.get(f"http://127.0.0.1:{port}/health", timeout=1).ok:
            break
    except requests.ConnectionError:
        time.sleep(0.2)
(BASE / "deploy" / ".port").write_text(str(port))
print(f"✅ 小助教 API 開好了：http://127.0.0.1:{port}")
print("   GET  /health")
print("   POST /review")
print("   GET  /threads/{thread_id}")

In [ ]:
#@title ▶ 執行：用 HTTP 送作業給小助教，再查存檔
thread_id = "小明-期中"  #@param {type:"string"}
作業 = "命名不一致"  #@param ["命名不一致", "變數打錯字", "正確的程式", "縮排亂掉"]
import sys; sys.path.insert(0, "/content/review-bot")
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
import json, requests

port_檔 = BASE / "deploy" / ".port"
if not port_檔.exists():
    raise SystemExit("⚠️ 服務還沒開：先執行上一格")
網址 = f"http://127.0.0.1:{port_檔.read_text()}"
try:
    回應 = requests.post(f"{網址}/review", json={"thread_id": thread_id, "code": 取得作業(作業)}, timeout=600)
except requests.ConnectionError:
    raise SystemExit("⚠️ 服務沒有在跑（可能重新啟動過執行階段）：先執行上一格")

print(f"POST /review → HTTP {回應.status_code}")
print(json.dumps(回應.json(), ensure_ascii=False, indent=2))
查詢 = {}
for tid in [thread_id, "還沒交作業的同學"]:
    查詢[tid] = requests.get(f"{網址}/threads/{tid}", timeout=30).json()
    print(f"\nGET /threads/{tid} → 有存檔：{查詢[tid]['有存檔']}，下一站：{查詢[tid]['下一站'] or ('（結束）' if 查詢[tid]['有存檔'] else '—')}")
記下(f"2-E 練習 1：API 服務（thread_id={thread_id}）", {"review": 回應.json(), "threads": 查詢})

### 👀 你看到了什麼

- 你沒有直接呼叫任何 Python 函式，只是「送一個網址請求」，就拿到回饋了。網頁、LINE bot 也都是這樣用它。
- 「小明-期中」有存檔、「還沒交作業的同學」沒有：**一個 thread_id 一張訂單紙**，跟 2-D 一樣。
- 存檔現在放在 Colab 這台機器的 SQLite 檔案裡。雲端的機器隨時會被換掉，所以上線時要換成存在資料庫裡的 checkpointer（例如 PostgresSaver）。
- 投影片 48 的三種上雲端做法（LangSmith Deployment、自己的容器如 Cloud Run、雲端廠商的 agent 執行環境），真正在跑的都是這一層。

---
## 練習 2：200 份同時批：Send，以及它的成本

> 📍 **對應簡報**：投影片 50「200 份同時批：Send」

第 2 堂說過：一份作業開一個子 agent、一張乾淨的桌子，才不會把第 148 份同學的名字寫進第 150 份。
LangGraph 的做法是 **Send**：條件邊回傳一串 `Send("批一份", 一份作業)`，「批一份」這個節點就會同時開很多個。
`max_concurrency` 限制最多同時跑幾份（模型 API 每分鐘有上限，一次衝 200 個會被擋）。

對照組是「一張桌子批全部」：只問模型一次，把所有作業塞在同一個 prompt 裡。

### 🤔 預測題
1. 批 5 份作業，平行版會呼叫模型幾次？一張桌子版呢？
2. 哪一種用的 token 比較多？為什麼？

In [ ]:
%%writefile /content/review-bot/deploy/batch.py
"""練習 2：用 Send 平行批改（map-reduce），並算一算成本。

START ─分派→ 批一份 × N（同時跑，每份一張乾淨的桌子）→ 彙整 → END

- Send：在條件邊裡說「批一份這個節點，照這份清單一份開一個」
- 結果們用 Annotated[list, operator.add]：每個「批一份」交回來的結果往下接著寫
- config={"max_concurrency": 2}：最多同時跑幾份，其餘排隊（模型 API 每分鐘有上限）
"""

import operator
from typing import Annotated, TypedDict

from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send

from skill_loader import 列出名片, 讀內文


class 批次State(TypedDict):
    作業們: list[dict]                            # [{"學生": "學生1", "code": "..."}, ...]
    結果們: Annotated[list[dict], operator.add]   # 往下接著寫，不會互相覆蓋


class 一份State(TypedDict):
    學生: str
    code: str


def _手冊() -> str:
    return 讀內文(列出名片(["friendly-feedback"])[0])


def 建立批改流程(model):
    手冊 = _手冊()

    def 分派(state: 批次State) -> list[Send]:
        # 每份作業開一個「批一份」：各自一張乾淨的桌子，不會把別人的作業看進來
        return [Send("批一份", {"學生": a["學生"], "code": a["code"]}) for a in state["作業們"]]

    def 批一份(state: 一份State) -> dict:
        回饋 = model.invoke([
            SystemMessage(手冊),
            HumanMessage(f"請批改這份作業：\n```python\n{state['code']}\n```"),
        ]).text
        return {"結果們": [{"學生": state["學生"], "回饋": 回饋.strip()}]}

    def 彙整(state: 批次State) -> dict:
        return {}

    builder = StateGraph(批次State)
    builder.add_node("批一份", 批一份)
    builder.add_node("彙整", 彙整)
    builder.add_conditional_edges(START, 分派, ["批一份"])
    builder.add_edge("批一份", "彙整")
    builder.add_edge("彙整", END)
    return builder.compile()


def 一次批全部(model, 作業們: list[dict], callbacks=None) -> str:
    """對照組：一張桌子批全部（便宜，但第 150 份時桌上堆滿前面 149 份）。"""
    全部 = "\n\n".join(f"### {a['學生']}\n```python\n{a['code']}\n```" for a in 作業們)
    return model.invoke(
        [SystemMessage(_手冊()), HumanMessage(f"請逐一批改下面每一位學生的作業：\n\n{全部}")],
        config={"callbacks": callbacks or []},
    ).text


class 用量計(BaseCallbackHandler):
    """數模型被呼叫幾次、加總 token（Colab 內建模型的 token 是估算值）。"""

    def __init__(self):
        self.次數 = 0
        self.input_tokens = 0
        self.output_tokens = 0

    def on_llm_end(self, response, **kwargs):
        self.次數 += 1
        for generations in response.generations:
            for g in generations:
                usage = getattr(getattr(g, "message", None), "usage_metadata", None) or {}
                self.input_tokens += usage.get("input_tokens", 0)
                self.output_tokens += usage.get("output_tokens", 0)


In [ ]:
#@title ▶ 執行：用 Send 平行批改，再和「一張桌子批全部」比成本
份數 = 5  #@param {type:"slider", min:2, max:8, step:1}
同時批幾份 = 2  #@param {type:"slider", min:1, max:5, step:1}
import sys; sys.path.insert(0, "/content/review-bot")
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
import time
from deploy import batch

範例們 = [v for k, v in 範例作業.items() if k != "空白"]
作業們 = [{"學生": f"學生{i + 1}", "code": 範例們[i % len(範例們)]} for i in range(份數)]

流程 = batch.建立批改流程(model)
顯示流程圖(流程)

平行 = batch.用量計()
t0 = time.time()
結果 = 流程.invoke({"作業們": 作業們, "結果們": []}, config={"max_concurrency": 同時批幾份, "callbacks": [平行]})
秒數 = time.time() - t0
for r in 結果["結果們"]:
    print(f"【{r['學生']}】{r['回饋'][:70].replace(chr(10), ' ')}…")

一張桌子 = batch.用量計()
batch.一次批全部(model, 作業們, callbacks=[一張桌子])

print(f"\n{'做法':<18}{'呼叫模型':>8}{'input token':>14}{'output token':>14}")
for 名稱, 計 in [("Send 平行（一份一桌）", 平行), ("一張桌子批全部", 一張桌子)]:
    print(f"{名稱:<18}{計.次數:>8}{計.input_tokens:>14,}{計.output_tokens:>14,}")
總平行 = 平行.input_tokens + 平行.output_tokens
總一桌 = 一張桌子.input_tokens + 一張桌子.output_tokens
print(f"\n平行版花了「一張桌子」的 {總平行 / max(總一桌, 1):.1f} 倍 token；平行跑 {份數} 份花了 {秒數:.1f} 秒（同時最多 {同時批幾份} 份）")
print("（Colab 內建模型的 token 是估算值）")
記下(f"2-E 練習 2：Send 平行批 {份數} 份", {"平行": vars(平行), "一張桌子": vars(一張桌子)})

### 👀 你看到了什麼

- 平行版每份作業呼叫一次模型（`份數` 次），一張桌子版只呼叫 1 次。
- 平行版的 input token 比較多：**每一份都要重新把手冊放上桌**。真實的多 agent 系統還要加上工具說明、互相傳話、摘要結果，
  Anthropic 的數字是單一 agent 的 **3–10 倍**。這裡的倍數只是小規模示範。
- 那為什麼還要平行？因為一張桌子塞 200 份，前面的作業會干擾後面的（context 污染），而且平行比較快。
  **先量一份花多少、再乘以 200，確定付得起再開。** 把「同時批幾份」調大，平行版會變快，但 token 一樣多。

---
## 練習 3：寄信前一定停下來：interrupt()

> 📍 **對應簡報**：投影片 51「寄信前一定停下來：interrupt()」

寄信不可逆，寄錯了收不回來。`interrupt()` 會把 State 存進 checkpointer 然後**停住**，把要寄的內容交給人看；
人看過之後，用 `Command(resume="寄")` 帶著**同一個 thread_id** 叫它繼續。等一小時、等三天都可以。

這裡的 `send_email` 是假的，只會把信寫進 `/content/review-bot/寄件匣.txt`。

### 🤔 預測題
1. 執行第一格之後，信寄出去了嗎？存檔裡的「下一站」是什麼？
2. 第二格選「不要寄」，寄件匣裡會有這封信嗎？

In [ ]:
%%writefile /content/review-bot/deploy/email_flow.py
"""練習 3：寄信前一定停下來，等人按確認（interrupt）。

START → 寫回饋 → 寄信（interrupt：停住，等人確認）→ END

interrupt() 會把 State 存進 checkpointer 然後停住；人看過內容之後，
用 Command(resume="寄") 帶著同一個 thread_id 叫它繼續，resume 的值就是 interrupt() 的回傳值。
"""

import datetime as _dt
from pathlib import Path
from typing import TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from langgraph.types import interrupt

from skill_loader import 列出名片, 讀內文

寄件匣 = Path(__file__).resolve().parent.parent / "寄件匣.txt"


class 寄信State(TypedDict, total=False):
    學生: str
    email: str
    code: str
    feedback: str
    已寄出: bool
    寄件紀錄: str


def send_email(to: str, body: str) -> str:
    """假的寄信工具：只寫進 寄件匣.txt。真實系統這裡會是 MCP 的寄信工具，寄出去就收不回來。"""
    with 寄件匣.open("a", encoding="utf-8") as f:
        f.write(f"[{_dt.datetime.now():%H:%M:%S}] 寄給 {to}\n{body}\n\n")
    return f"（模擬）已寄給 {to}"


def 建立寄信流程(model, checkpointer):
    手冊 = 讀內文(列出名片(["friendly-feedback"])[0])

    def 寫回饋(state: 寄信State) -> dict:
        回饋 = model.invoke([
            SystemMessage(手冊),
            HumanMessage(f"請批改這份作業：\n```python\n{state['code']}\n```"),
        ]).text
        return {"feedback": 回饋.strip()}

    def 寄信(state: 寄信State) -> dict:
        # 寄信不可逆：先停下來，把要寄的內容交給人看
        答案 = interrupt({"收件人": state["email"], "內容": state["feedback"]})
        if 答案 == "寄":
            return {"已寄出": True, "寄件紀錄": send_email(state["email"], state["feedback"])}
        return {"已寄出": False, "寄件紀錄": f"確認的人回答「{答案}」，這封信沒有寄出"}

    builder = StateGraph(寄信State)
    builder.add_node("寫回饋", 寫回饋)
    builder.add_node("寄信", 寄信)
    builder.add_edge(START, "寫回饋")
    builder.add_edge("寫回饋", "寄信")
    builder.add_edge("寄信", END)
    return builder.compile(checkpointer=checkpointer)


In [ ]:
#@title ▶ 執行（第 1 步）：寫好回饋，停在寄信前面等確認
thread_id = "小明-寄信"  #@param {type:"string"}
收件人 = "xiaoming@example.com"  #@param {type:"string"}
import sys; sys.path.insert(0, "/content/review-bot")
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
import dual_core
from deploy import email_flow

流程 = email_flow.建立寄信流程(model, dual_core.開存檔(BASE / "deploy" / "email_checkpoints.sqlite"))
config = {"configurable": {"thread_id": thread_id}}
結果 = 流程.invoke({"學生": "小明", "email": 收件人, "code": 取得作業("變數打錯字")}, config)

if "__interrupt__" in 結果:
    待確認 = 結果["__interrupt__"][0].value
    print("⏸️  停在寄信前面，等人確認：")
    print("   收件人：", 待確認["收件人"])
    print("   內容：")
    print("   " + 待確認["內容"].replace("\n", "\n   "))
print("\n存檔裡的下一站：", 流程.get_state(config).next)
寄出了嗎 = email_flow.寄件匣.exists() and 收件人 in email_flow.寄件匣.read_text(encoding="utf-8")
print("寄件匣裡有這封信嗎？", "有" if 寄出了嗎 else "沒有（還沒寄）")
記下(f"2-E 練習 3：停在寄信前（thread_id={thread_id}）", {"下一站": list(流程.get_state(config).next)})

In [ ]:
#@title ▶ 執行（第 2 步）：人看過了，決定寄不寄
thread_id = "小明-寄信"  #@param {type:"string"}
決定 = "寄"  #@param ["寄", "不要寄"]
import sys; sys.path.insert(0, "/content/review-bot")
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
from langgraph.types import Command
import dual_core
from deploy import email_flow

流程 = email_flow.建立寄信流程(model, dual_core.開存檔(BASE / "deploy" / "email_checkpoints.sqlite"))
config = {"configurable": {"thread_id": thread_id}}
if not 流程.get_state(config).next:
    print(f"thread_id「{thread_id}」沒有在等確認的信：先執行第 1 步，或檢查 thread_id 有沒有打錯。")
else:
    結果 = 流程.invoke(Command(resume=決定), config)     # resume 的值就是 interrupt() 的回傳值
    print("📮", 結果["寄件紀錄"])
    print("已寄出：", 結果["已寄出"])
    if email_flow.寄件匣.exists():
        print("\n寄件匣.txt 最後幾行：")
        print("\n".join(email_flow.寄件匣.read_text(encoding="utf-8").splitlines()[-6:]))
    記下(f"2-E 練習 3：決定「{決定}」（thread_id={thread_id}）", {"已寄出": 結果["已寄出"], "寄件紀錄": 結果["寄件紀錄"]})

### 👀 你看到了什麼

- 第 1 步之後信**還沒寄**，下一站停在 `寄信`：流程停住了，State 安全地存在存檔裡。
- 第 2 步選「寄」才會寫進寄件匣；選「不要寄」就不會。第 2 堂說的「寄之前一定要有人按確認」，在這裡從一句話變成了一行 `interrupt()`。
- 幹部可能三天後才按確認，中間雲端機器早就換過了，所以上線一定要用不會忘記的 checkpointer（Postgres 這類資料庫）。

---
## 練習 4：上線前檢查，順便產生部署檔

> 📍 **對應簡報**：投影片 46（金鑰放 Secret Manager）、47（langgraph.json 是出貨單）、52「上線之後：看得見、關得住」

下面幾格會寫出上雲端要用的檔案（**只產生檔案，不會真的 build 或部署**）：

| 檔案 | 是什麼 |
| --- | --- |
| `langgraph.json` | 出貨單：告訴 LangSmith Deployment / `langgraph dev`「graph 在哪個檔案的哪個變數」 |
| `deploy/app_graph.py` | 出貨單指向的進入點，模型金鑰從環境變數讀 |
| `requirements.txt` | 要裝哪些套件（版本範圍釘死） |
| `Dockerfile` | 把程式、`skills/`、`scripts/` 裝進同一個容器的食譜 |
| `.dockerignore` | 不准裝進容器的東西：`.env`、存檔、紀錄 |
| `deploy/preflight.py` | 上線前檢查：每一項都用程式檢查，不靠感覺 |

### 🤔 預測題
上線前檢查有 8 項，你猜哪幾項會是 ⚠️（還要處理）？

In [ ]:
%%writefile /content/review-bot/langgraph.json
{
  "dependencies": ["."],
  "graphs": {
    "review_bot": "./deploy/app_graph.py:graph"
  },
  "env": ".env"
}


In [ ]:
%%writefile /content/review-bot/deploy/app_graph.py
"""langgraph.json 指向的進入點：LangSmith Deployment / `langgraph dev` 會 import 這個檔案裡的 graph。

部署平台會自己提供 checkpointer（例如 Postgres），所以這裡不用 開存檔()。
模型來源用環境變數：WORKSHOP_PROVIDER=gemini，金鑰 GOOGLE_API_KEY 由平台的 Secret 注入。
"""

import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
os.environ.setdefault("WORKSHOP_PROVIDER", "gemini")

from dual_core import 建立雙核小助教  # noqa: E402
from llm import get_model  # noqa: E402

graph = 建立雙核小助教(get_model())


In [ ]:
%%writefile /content/review-bot/requirements.txt
# 作業小助教上雲端需要的套件（版本範圍和 notebook 一致）
langgraph>=1.1,<2
langchain>=1.2,<2
langchain-google-genai>=4,<5
langgraph-checkpoint-sqlite>=3,<4
fastapi>=0.115
uvicorn>=0.30
# 正式環境把 SQLite 換成 Postgres 時再加：langgraph-checkpoint-postgres


In [ ]:
%%writefile /content/review-bot/Dockerfile
# 把作業小助教（程式 ＋ skills/ ＋ scripts/）裝進同一個容器
# 在 review-bot/ 資料夾裡：
#   docker build -t review-bot .
#   docker run -p 8080:8080 -e GOOGLE_API_KEY=你的key review-bot
# 金鑰不要寫進這個檔案：部署時由雲端的 Secret Manager 變成環境變數注入
FROM python:3.12-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 程式、手冊、腳本一起進映像：手冊和程式永遠是同一個版本
COPY . .

ENV WORKSHOP_PROVIDER=gemini \
    PORT=8080
CMD ["sh", "-c", "uvicorn deploy.server:app --host 0.0.0.0 --port ${PORT}"]


In [ ]:
%%writefile /content/review-bot/.dockerignore
# 不要裝進容器的東西：金鑰、本機存檔、執行紀錄
.env
.llm.json
checkpoints.sqlite*
history.jsonl
寄件匣.txt
2B_results.json
__pycache__/


In [ ]:
%%writefile /content/review-bot/deploy/preflight.py
"""練習 4：上線前檢查。每一項都是用程式檢查（sensor），不是靠感覺。

用法：python -m deploy.preflight（在 review-bot/ 資料夾裡）
"""

import os
import re
import sys
from pathlib import Path

BASE = Path(__file__).resolve().parent.parent

金鑰樣式 = [
    (r"AIza[0-9A-Za-z\-_]{35}", "Google API key"),
    (r"sk-[A-Za-z0-9_\-]{20,}", "OpenAI / Anthropic 類型的 key"),
    (r"(?i)(api[_-]?key|token|password)\s*=\s*['\"][^'\"\s]{8,}['\"]", "寫死在程式裡的密碼"),
]
可疑樣式 = [
    (r"curl[^\n|]*\|\s*(ba)?sh|wget[^\n|]*\|\s*(ba)?sh", "下載後直接執行（curl … | bash）"),
    (r"base64\s+(-d|--decode)", "解碼一段看不懂的內容"),
    (r"(?i)ignore (all |the )?previous|忽略(之前|以上|前面)", "要 AI 忽略之前的指示"),
    (r"(?i)環境變數|read_env|WORKSHOP_API_KEY|\$[A-Z_]*KEY", "要 AI 去讀金鑰或環境變數"),
]
文字檔 = {".py", ".md", ".sh", ".json", ".txt", ".toml", ".yaml", ".yml", ""}
略過 = {"__pycache__", ".llm.json", "history.jsonl", ".env.example"}


def _檔案們(root: Path):
    for p in sorted(root.rglob("*")):
        if p.is_file() and p.suffix in 文字檔 and not (set(p.parts) & 略過) and p.name not in 略過:
            yield p


def _掃(root: Path, 樣式) -> list[str]:
    找到 = []
    for p in _檔案們(root):
        if p.resolve() == Path(__file__).resolve():
            continue  # 不掃自己（這個檔案裡本來就寫著這些樣式）
        for i, line in enumerate(p.read_text(encoding="utf-8", errors="ignore").splitlines(), 1):
            for pattern, 說明 in 樣式:
                if re.search(pattern, line):
                    找到.append(f"{p.relative_to(BASE)} 第 {i} 行：{說明}")
    return 找到


def 檢查() -> list[tuple[str, str, str]]:
    """回傳 [(結果, 項目, 說明)]，結果是 ✅ / ⚠️ / ❌。"""
    結果 = []

    外洩 = _掃(BASE, 金鑰樣式)
    結果.append(("❌" if 外洩 else "✅", "程式和手冊裡沒有寫死的金鑰",
               "；".join(外洩) if 外洩 else "金鑰應該放在 Secrets / Secret Manager，從環境變數讀"))

    來源 = os.environ.get("WORKSHOP_PROVIDER") or "（看 0. 環境設定 的選擇）"
    有key = bool(os.environ.get("GOOGLE_API_KEY"))
    結果.append(("✅" if 有key else "⚠️", "模型金鑰從環境變數來",
               f"WORKSHOP_PROVIDER={來源}；GOOGLE_API_KEY {'已設定' if 有key else '沒設定（Colab 內建模型不需要，上雲端一定要）'}"))

    可疑 = _掃(BASE / "skills", 可疑樣式)
    結果.append(("❌" if 可疑 else "✅", "第三方 Skill 掃描（2-C 的三個習慣）",
               "；".join(可疑) if 可疑 else "skills/ 裡沒有可疑的指令"))

    sys.path.insert(0, str(BASE))
    from tools.run_code import 執行作業

    擋下 = 執行作業("import os\nos.listdir('.')").startswith("⛔")
    結果.append(("✅" if 擋下 else "❌", "run_code 有圍欄", "import os 會被擋下" if 擋下 else "import os 沒有被擋下！"))

    dual = (BASE / "dual_core.py").read_text(encoding="utf-8")
    結果.append(("✅" if "recursion_limit" in dual else "❌", "agent 有 recursion_limit", "裡面的小 agent 最多跑 25 步"))
    結果.append(("⚠️" if "SqliteSaver" in dual else "✅", "checkpointer 是持久的",
               "SQLite 只存在這一台機器；雲端機器會被換掉，上線要改用 PostgresSaver"))

    寄信 = BASE / "deploy" / "email_flow.py"
    有確認 = 寄信.exists() and "interrupt(" in 寄信.read_text(encoding="utf-8")
    結果.append(("✅" if 有確認 else "❌", "不可逆的動作（寄信）前面有 interrupt", "寄信前會停下來等人確認" if 有確認 else "找不到 interrupt"))

    缺 = [f for f in ("langgraph.json", "requirements.txt", "Dockerfile", ".dockerignore") if not (BASE / f).exists()]
    結果.append(("❌" if 缺 else "✅", "部署檔都在", f"缺少：{'、'.join(缺)}" if 缺 else "langgraph.json、requirements.txt、Dockerfile、.dockerignore"))
    return 結果


def 印出檢查結果() -> bool:
    結果 = 檢查()
    for 符號, 項目, 說明 in 結果:
        print(f"{符號} {項目}\n     {說明}")
    失敗 = sum(1 for r in 結果 if r[0] == "❌")
    print(f"\n{'🚫 有 ' + str(失敗) + ' 項沒過，先修好再上線' if 失敗 else '🎉 沒有 ❌，⚠️ 的項目上線前記得處理'}")
    return 失敗 == 0


if __name__ == "__main__":
    sys.exit(0 if 印出檢查結果() else 1)


In [ ]:
#@title ▶ 執行：上線前檢查
import sys; sys.path.insert(0, "/content/review-bot")
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
from deploy import preflight

通過 = preflight.印出檢查結果()
記下("2-E 練習 4：上線前檢查", [{"結果": a, "項目": b, "說明": c} for a, b, c in preflight.檢查()])

### 👀 你看到了什麼

- ✅ 的是程式已經幫你守住的：沒有寫死的金鑰、手冊裡沒有可疑指令、`run_code` 有圍欄、agent 有 `recursion_limit`、寄信前有 `interrupt`、部署檔都在。
- ⚠️ 的是上雲端前還要處理的：
  - **金鑰**：Colab 內建模型不用 key，但雲端一定要用正式的 API key，放進雲端的 Secret Manager（Colab 左邊 🔑「密鑰」就是它的縮小版）。
  - **checkpointer**：SQLite 只存在這一台機器，上線要換成 PostgresSaver。
- 想再試一次 2-C：把 `detailed-feedback` 那本有問題的手冊放進 `skills/`，再跑一次檢查，看「第三方 Skill 掃描」會不會變成 ❌。

### 真的要部署的時候（課後參考，會產生費用，指令以各平台官方文件為準）

```bash
# 在 review-bot/ 資料夾裡
# 1. 本機先跑通（需要 pip install "langgraph-cli[inmem]"）
langgraph dev

# 2a. 自己的容器：打包成映像，放到 Cloud Run 這類服務；金鑰用 Secret Manager 注入
docker build -t review-bot .
gcloud run deploy review-bot --source . --set-secrets=GOOGLE_API_KEY=review-bot-key:latest

# 2b. 代管：把這個資料夾推上 GitHub，在 LangSmith Deployment 選這個 repo 部署（它會讀 langgraph.json）
```

上雲端不是換個地方跑，是把 harness 補完整：**每一步看得見（trace、eval、成本），每一個權限關得住（最小權限、沙箱、審 Skill、金鑰、人工確認）。**

## 收尾：存檔與交作業

- 💾 **存到雲端硬碟**：把 `/content/review-bot/`（你改過的手冊、程式、State 紀錄）存一份到自己的 Google 雲端硬碟。
- ⬇️ **下載 zip**：直接下載到電腦。
- 📸 **最後一格**：印出這本 notebook 跑過的所有 State，截圖就能交作業。

Colab 斷線後 `/content/` 裡的檔案會消失，想留下來就先存。notebook 本身請用上方選單「檔案 → 在雲端硬碟中儲存副本」。

In [ ]:
#@title 💾 存到 Google 雲端硬碟（會跳出授權視窗）
import sys; sys.path.insert(0, "/content/review-bot")
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
存到雲端硬碟()

In [ ]:
#@title ⬇️ 下載 zip
import sys; sys.path.insert(0, "/content/review-bot")
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
下載壓縮檔()

In [ ]:
#@title 📸 印出整份 State 歷程（截圖交作業）
import sys; sys.path.insert(0, "/content/review-bot")
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
印出歷程()